In [ ]:
# @title ▶️ CLICK PLAY TO START
# @markdown ### 👈 Click the Play (▶️) button on the left to start FreeFakeStudio!
# @markdown *(Everything is automated. When ready, a big button will appear below to open your studio!)*

import os, sys, subprocess, time, re, glob
import torch
from IPython.display import display, HTML, clear_output

if not torch.cuda.is_available():
    clear_output()
    print("❌ GPU required. Please click: Runtime -> Change runtime type -> T4 GPU -> Save, then click Play again.")
    raise SystemExit(1)

print("⏳ Starting FreeFakeStudio... (~1-2 min)")

# ── 1. ComfyUI Core ─────────────────────────────────────────
if not os.path.exists('/content/ComfyUI'):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", "/content/ComfyUI"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/requirements.txt"], check=True)

# ── 2. Studio Utilities & Dependencies ──────────────────────
subprocess.run(["apt-get", "update", "-qq"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rembg[gpu]", "onnxruntime-gpu", "aria2", "gradio"], check=True)

# ── 3. Pre-download Fast Default Model (Z-Image Turbo) ──────
DIFF   = '/content/ComfyUI/models/diffusion_models'
CLIP   = '/content/ComfyUI/models/clip'
TXTENC = '/content/ComfyUI/models/text_encoders'
VAE    = '/content/ComfyUI/models/vae'
for d in [DIFF, CLIP, TXTENC, VAE]:
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, name):
    path = os.path.join(dest_dir, name)
    if not (os.path.exists(path) and os.path.getsize(path) > 1024):
        if os.path.exists(path):
            os.remove(path)
        subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url, "-d", dest_dir, "-o", name], check=True)

dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors', DIFF, 'z-image-turbo-fp8-e4m3fn.safetensors')
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors', CLIP, 'qwen_3_4b.safetensors')
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors', VAE, 'ae.safetensors')

for f in glob.glob(f'{CLIP}/*'):
    link = os.path.join(TXTENC, os.path.basename(f))
    if not os.path.exists(link):
        try: os.symlink(f, link)
        except: pass
for f in glob.glob(f'{TXTENC}/*'):
    link = os.path.join(CLIP, os.path.basename(f))
    if not os.path.exists(link):
        try: os.symlink(f, link)
        except: pass

# ── 4. Pull FreeFakeStudio & Start Backend ──────────────────
REPO = 'https://github.com/MuntasirMalek/FreeFakeStudio.git'
if os.path.exists('/content/FreeFakeStudio'):
    os.chdir('/content/FreeFakeStudio')
    subprocess.run(["git", "pull", "--ff-only"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/FreeFakeStudio"], check=True, stdout=subprocess.DEVNULL)
    os.chdir('/content/FreeFakeStudio')

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(
    [sys.executable, "-u", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

gradio_url = None
url_pattern = re.compile(r'(https://[a-zA-Z0-9-]+\.gradio\.live)')

for line in proc.stdout:
    match = url_pattern.search(line)
    if match:
        gradio_url = match.group(1)
        break

if gradio_url:
    web_app_url = f"https://muntasirmalek.github.io/FreeFakeStudio/?backend={gradio_url}"
    clear_output()
    display(HTML(f'''
    <div style="text-align: center; margin: 20px 0; padding: 32px 20px; background: #ecfdf5; border: 3px solid #10b981; border-radius: 20px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;">
        <h2 style="color: #065f46; margin: 0 0 10px 0; font-size: 26px; font-weight: 800;">🎉 Studio is Ready!</h2>
        <p style="color: #047857; margin: 0 0 24px 0; font-size: 15px;">Click the button below to open your studio:</p>
        <a href="{web_app_url}" target="_blank" style="display: inline-block; background: #10b981; color: #ffffff; padding: 18px 56px; border-radius: 16px; font-size: 24px; font-weight: 800; text-decoration: none; box-shadow: 0 10px 25px rgba(16,185,129,0.35); letter-spacing: 0.5px;">
            OPEN STUDIO
        </a>
        <div style="margin-top: 24px; font-size: 13px; color: #047857;">
            <span>Direct Public Link: </span>
            <a href="{gradio_url}" target="_blank" style="color: #059669; font-weight: 700; text-decoration: underline;">{gradio_url}</a>
        </div>
    </div>
    '''))

try:
    for line in proc.stdout:
        pass
except KeyboardInterrupt:
    proc.terminate()
